# **Utility Functions**

This notebook is dedicated to writing the utility functions needed to pass to the optimiser from the `SatelliteSystem` object that handles the sceduling. Let us first create the satellite system,

In [56]:
# Importing,
from Scheduler import SatelliteSystem, Satellite, Station
import numpy as np

# Creating the system,
system = SatelliteSystem(planetary_radius=6000, planetary_mass=5.972e24, planetary_angular_velocity=7.2921159e-5)

# Adding satellites,
sat1 = Satellite(name="LEO-INTER-1", altitude=500, phase_angle=0, orbit_prograde=True)
sat2 = Satellite(name="LEO-INTER-2", altitude=500, phase_angle=np.pi/2, orbit_prograde=True)
system.add_sat(sat1)
system.add_sat(sat2)

# Adding ground stations,
station1 = Station(name="G1", position=np.pi/3)
station2 = Station(name="G2", position=2*np.pi/3)
station3 = Station(name="G3", position=4*np.pi/3)
system.add_station(station1)
system.add_station(station2)
system.add_station(station3)

# Performing the calculations,
transfers = system.compute_visibilities(station1, interval=(0, 6*60*60)) # 6 hours in seconds.
transfers

{'LEO-INTER-1': [('00:09:44', '00:20:54'),
  ('01:42:21', '01:53:31'),
  ('03:14:57', '03:26:08'),
  ('04:47:12', '04:58:44')],
 'LEO-INTER-2': [('01:19:17', '01:30:27'),
  ('02:51:32', '03:03:04'),
  ('04:24:09', '04:35:19'),
  ('05:56:45', '06:00:00')]}

We will write a function to calculate all transfer window lengths from a given staton $G$, 

In [35]:
def _compute_transfer_time(start_end):
    """Computes the transfer time in seconds given a start and end time in the format HH:MM:SS where
    start_end is the tuple [start_time, end_time]."""
    delta_HH = int(start_end[1][0:2]) - int(start_end[0][0:2])
    delta_MM = int(start_end[1][3:5]) - int(start_end[0][3:5])
    delta_SS = int(start_end[1][6:8]) - int(start_end[0][6:8])
    T = 3600*delta_HH + 60*delta_MM + delta_SS

    return T

def _time_to_seconds(time_str):
    """Convert a time string in the format HH:MM:SS to seconds."""
    HH, MM, SS = map(int, time_str.split(':'))
    return 3600*HH + 60*MM + SS

def intervals_overlap(a, b):
    """Return True if the two intervals overlap."""
    a_start = _time_to_seconds(a[0])
    a_end   = _time_to_seconds(a[1])
    b_start = _time_to_seconds(b[0])
    b_end   = _time_to_seconds(b[1])
    return a_start < b_end and b_start < a_end

Now we want to find the set $\mathcal{O}$ all indice tuples of overlapping windows,

In [55]:
# Exracting all transfer windows into a single list,
transfer_windows = []
for sat_name, windows in transfers.items():
    for window in windows:
        transfer_windows.append(window)

# Finding overlapping windows,
O = []
for i in range(len(transfer_windows)):
    for j in range(i+1, len(transfer_windows)):
        a = transfer_windows[i]
        b = transfer_windows[j]
        if intervals_overlap(a, b):
            print(f"Overlap between {a} and {b}")
            O.append((a, b))